# 00 · Exploration
Profiling and one-off diagnostics behind the pipeline decisions.
Not part of the scheduled job; run after the pipeline has populated all layers.

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

from easymoney_pipeline.transforms import add_customer_timeline, align_sales_to_snapshot

BRONZE = "workspace.easymoney_bronze"
SILVER = "workspace.easymoney_silver"
GOLD = "workspace.easymoney_gold"
KEYS = ["customer_id", "partition_month"]
TOTAL = Window.partitionBy()

## Bronze: grain of the customer tables
`rows = customer_month_pairs` in all three tables → one row per customer and month, 17 monthly snapshots.

In [0]:
%sql
SELECT 'customer_products' AS source_table,
       COUNT(*) AS rows,
       COUNT(DISTINCT pk_cid, pk_partition) AS customer_month_pairs,
       COUNT(DISTINCT pk_cid) AS customers,
       COUNT(DISTINCT pk_partition) AS months,
       MIN(pk_partition) AS first_month,
       MAX(pk_partition) AS last_month
FROM workspace.easymoney_bronze.customer_products
UNION ALL
SELECT 'customer_commercial_activity', COUNT(*), COUNT(DISTINCT pk_cid, pk_partition),
       COUNT(DISTINCT pk_cid), COUNT(DISTINCT pk_partition), MIN(pk_partition), MAX(pk_partition)
FROM workspace.easymoney_bronze.customer_commercial_activity
UNION ALL
SELECT 'customer_sociodemographics', COUNT(*), COUNT(DISTINCT pk_cid, pk_partition),
       COUNT(DISTINCT pk_cid), COUNT(DISTINCT pk_partition), MIN(pk_partition), MAX(pk_partition)
FROM workspace.easymoney_bronze.customer_sociodemographics

## Bronze: null profile
Findings: `payroll` / `pension_plan` 61 NULLs (same rows) · `salary` ~26% · `region_code` 2,264 ·
`entry_channel` 133,033 · `segment` 133,944.

In [0]:
for t in ["customer_products", "customer_sociodemographics", "customer_commercial_activity"]:
    df = spark.table(f"{BRONZE}.{t}").drop("_c0", "_source_file", "_ingested_at")
    print(t)
    display(df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]))

## Silver: distributions and consistency
Findings: age 2–105 (minors excluded from the propensity population) · `region_code` NULL = 2,252 non-ES + 12 ES ·
channel and segment NULLs overlap in 132,423 rows · no `entry_month` after `partition_month`.

In [0]:
socio = spark.table(f"{SILVER}.customer_sociodemographics")
activity = spark.table(f"{SILVER}.customer_commercial_activity")

display(socio.agg(
    F.min("age").alias("age_min"),
    F.max("age").alias("age_max"),
    F.round(F.avg(F.col("salary").isNull().cast("int")) * 100, 1).alias("pct_salary_null"),
))
display(socio.groupBy((F.col("country_id") == "ES").alias("is_spain"),
                      F.col("region_code").isNull().alias("region_null")).count())
display(activity.groupBy("segment_code", "segment_name").count().orderBy("segment_code"))
display(activity.groupBy(F.col("entry_channel").isNull().alias("channel_null"),
                         F.col("segment_code").isNull().alias("segment_null")).count())
print("entry_month > partition_month:",
      activity.filter(F.col("entry_month") > F.col("partition_month")).count())

In [0]:
sales = spark.table(f"{SILVER}.sales")
customers = spark.table(f"{SILVER}.customer_products").select("customer_id").distinct()

print("sales from customers with no monthly snapshot:",
      sales.join(customers, "customer_id", "left_anti").count())
display(sales.agg(
    F.min("sale_month").alias("first_month"),
    F.max("sale_month").alias("last_month"),
    F.min("net_margin").alias("margin_min"),
    F.max("net_margin").alias("margin_max"),
    F.sum((F.col("net_margin") < 0).cast("int")).alias("negative_margin_sales"),
))

## Gold: where the margin is
Findings: `pension_plan` is 8% of sales but 79% of net margin (~6,000 per sale vs ~70 for accounts) ·
July 2018 acquisition wave (252k → 339k customers, active rate 48% → 38%) ·
`short_term_deposit` discontinued from Feb 2019 · `em_account_p` / `em_account_pp` at 0% penetration.

In [0]:
display(spark.table(f"{GOLD}.kpi_sales_monthly")
    .groupBy("product_family")
    .agg(F.sum("n_sales").alias("n_sales"), F.round(F.sum("net_margin_total"), 0).alias("net_margin"))
    .withColumn("pct_sales", F.round(F.col("n_sales") / F.sum("n_sales").over(TOTAL) * 100, 1))
    .withColumn("pct_margin", F.round(F.col("net_margin") / F.sum("net_margin").over(TOTAL) * 100, 1))
    .orderBy(F.desc("net_margin")))

display(spark.table(f"{GOLD}.kpi_customers_monthly").orderBy("partition_month"))

## Target diagnosis: holdings flag vs actual sales
Every 0→1 transition of `pension_plan` (adult, non-deceased customers) split by prior holding and actual sale in t+1.
Result: first-time activations always come with a sale (18,451); reactivations never do (18,318).
Decision: target = actual sale in t+1, population = customers who never held the product.

In [0]:
cm = spark.table(f"{GOLD}.customer_monthly")
pension_ids = (spark.table(f"{SILVER}.product_description")
    .filter(F.col("product_name") == "pension_plan").select("product_id"))
sales_next = align_sales_to_snapshot(spark.table(f"{SILVER}.sales"), pension_ids)

transitions = (add_customer_timeline(cm, "pension_plan")
    .filter(F.col("has_next")
            & (F.col("pension_plan") == 0) & (F.col("next_flag") == 1)
            & ~F.col("is_deceased") & (F.col("age") >= 18))
    .join(sales_next, KEYS, "left")
    .fillna(0, subset=["has_sale_next_month"]))

display(transitions.groupBy("ever_had_before", "has_sale_next_month").count()
    .withColumn("pct", F.round(F.col("count") / F.sum("count").over(TOTAL) * 100, 1))
    .orderBy("ever_had_before", "has_sale_next_month"))

## Target rate by month
Positive rate 0.23–0.64% (≈1 in 300). Jan 2018 is an outlier (0.85%): no prior history to exclude past holders.

In [0]:
display(spark.table(f"{GOLD}.propensity_features")
    .groupBy("partition_month")
    .agg(F.count("*").alias("eligible_customers"),
         F.sum("target").alias("sales_t_plus_1"),
         F.round(F.avg("target") * 100, 3).alias("rate_pct"))
    .orderBy("partition_month"))